<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/HTSModelDataPrep26082026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# This script begins with tables exported from the National Data warehouse and
#concludes with the creation of ML-ready datasets that are  then read in by
# model training scripts. By ML-ready datasets, we mean datasets in which each
# row is an observation with a labeled outcome.

# The sript proceeds through three steps:
# 1) Combining and filtering original tables
# 2) Data Cleaning and feature generation
# 3) Missing data imputation

# First, load necessary packages
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from statsmodels.imputation.mice import MICEData


                       **  Combine and filter original tables**

In [2]:
# This script assumes user is accessing tables extracted and loaded from ODS Database.

#Load tables
eligibility = pd.read_csv('eligibility.csv', low_memory=False)
tests = pd.read_csv('tests.csv', low_memory=False)
xwalk = pd.read_csv('ActiveEMRSites_07152026.csv', low_memory=False)
clients = pd.read_csv('clients.csv', low_memory=False)

In [3]:
# Preview content of tables
rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

rows,cols = tests.shape
print(f"There are {rows} rows and {cols} columns in the tests table")

rows,cols = xwalk.shape
print(f"There are {rows} rows and {cols} columns in the xwalk table")

rows,cols = clients.shape
print(f"There are {rows} rows and {cols} columns in the clients table")

There are 764324 rows and 87 columns in the eligibility table
There are 1307055 rows and 34 columns in the tests table
There are 2456 rows and 19 columns in the xwalk table
There are 12199267 rows and 6 columns in the clients table


In [27]:
eligibility = pd.read_csv('eligibility.csv', low_memory=False)

In [28]:
rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

There are 2709027 rows and 87 columns in the eligibility table


In [5]:

# Tests
# Filter tests to results that are positive or negative (exclude inconclusive)
display(tests.columns)

tests['FinalTestResult']=tests['FinalTestResult'].str.upper()
tests = tests[tests['FinalTestResult'] != 'INCONCLUSIVE']
tests = tests[tests['FinalTestResult'] != 'INVALID']

# Select columns to keep and remove duplicates(We are keeping all columns)
cols_to_keep = ['SiteCode','PatientPk','EverTestedForHiv', 'MonthsSinceLastTest', 'FinalTestResult',
       'ClientTestedAs','CoupleDiscordant','PriorityPopulationType']
tests = tests[cols_to_keep]
tests = tests.drop_duplicates(subset=["SiteCode", "PatientPk"])

rows,cols = tests.shape
print(f"There are {rows} rows and {cols} columns in the tests table")


Index(['SiteCode', 'PatientPk', 'EverTestedForHiv', 'MonthsSinceLastTest',
       'FinalTestResult', 'ClientTestedAs', 'CoupleDiscordant',
       'PriorityPopulationType'],
      dtype='object')

There are 1198550 rows and 8 columns in the tests table


In [6]:
#Clients
# Select columns to keep and remove duplicates. We are keeping all columns in clients dataset
display(clients.columns)

clients = clients.drop_duplicates(subset= ["SiteCode","PatientPk"])

rows,cols = clients.shape
print(f"There are {rows} rows and {cols} columns in the clients table")

Index(['PatientPk', 'SiteCode', 'Dob', 'Sex', 'MaritalStatus',
       'PatientDisabled'],
      dtype='object')

There are 12199267 rows and 6 columns in the clients table


In [29]:
# Eligibility
# Convert visitdate from character to date and filter to between April 2025 and June 2026
print(eligibility["VisitDate"].dtype)
eligibility['VisitDate'] = pd.to_datetime(eligibility['VisitDate'].astype(str).str[:10])

eligibility = eligibility[eligibility['VisitDate'] >= '2025-04-01']
eligibility = eligibility[eligibility['VisitDate'] <= '2026-06-30']

rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

object
There are 2509280 rows and 87 columns in the eligibility table


In [30]:
# Select columns to keep and remove duplicates(Calculation of missingness and variance done separately)
cols = eligibility.columns
#print(cols)
cols_to_keep = ['SiteCode', 'PatientPk','VisitDate', 'PopulationType', 'KeyPopulation', 'PriorityPopulation',
                'IsHealthWorker','RelationshipWithContact', 'TestedHIVBefore','ResultOfHIV','EverHadSex',
                'SexuallyActive', 'NewPartner', 'PartnerHIVStatus', 'CoupleDiscordant','MultiplePartners', 'NumberOfPartners',
                'AlcoholSex', 'MoneySex','CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant','BreastfeedingMother',
                'ExperiencedViolenceScreening','CurrentlyOnPrep','TraditionalProcedures','MothersStatus',
                'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices','ViolenceScreeningType','Disability', 'DisabilityType']
eligibility = eligibility[cols_to_keep]
eligibility = eligibility.drop_duplicates(subset=["SiteCode", "PatientPk"])

rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

There are 2179395 rows and 35 columns in the eligibility table


In [31]:
eligibility.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'CoupleDiscordant',
       'MultiplePartners', 'NumberOfPartners', 'AlcoholSex', 'MoneySex',
       'CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType'],
      dtype='object')

In [10]:
xwalk.columns

Index(['MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agency', 'EMR', 'EMR_Status', 'HTS_Use',
       'HTS_Deployment', 'Project', 'LoadDate', 'InfrastructureType',
       'KEPH_Level', 'KMPDC_reg_no', 'Ward'],
      dtype='object')

In [11]:
# Select columns to keep and remove duplicates
cols = xwalk.columns

cols_to_keep =['MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agency', 'EMR','Project', 'KEPH_Level','Ward']
xwalk = xwalk[cols_to_keep]
xwalk = xwalk.drop_duplicates(subset=["MFL_Code"])

rows,cols = xwalk.shape
print(f"There are {rows} rows and {cols} columns in the xwalk table")

There are 2456 rows and 13 columns in the xwalk table


In [32]:
# Join the three tables on SiteCode and PatientPK
merged = pd.merge(eligibility, clients, on=["SiteCode", "PatientPk"], how="left")
merged = pd.merge(merged, tests, on=["SiteCode", "PatientPk"], how="left")
merged = pd.merge(merged, xwalk, left_on=["SiteCode"], right_on=["MFL_Code"], how="left")

rows,cols = merged.shape
print(f"There are {rows} rows and {cols} columns in the merged table")

#download the merged file
import requests
import csv
merged.to_csv('merged.csv', index=False)

There are 2179395 rows and 58 columns in the merged table


In [62]:
#Rename the merged dataset to hts
hts = merged.copy()


rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2179395 rows and 58 columns in the hts table


In [63]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'CoupleDiscordant_x',
       'MultiplePartners', 'NumberOfPartners', 'AlcoholSex', 'MoneySex',
       'CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType', 'Dob', 'Sex',
       'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs',
       'CoupleDiscordant_y', 'PriorityPopulationType', 'MFL_Code',
       'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Long

Quick Checks

In [64]:
#Filter the data to exclude ResultofHIV positive
hts['ResultOfHIV'] = hts['ResultOfHIV'].str.upper()
hts = hts[hts['ResultOfHIV'] != 'POSITIVE']

In [65]:
rows,cols = hts.shape
print(f" There are {rows} Rows and {cols} Columns in HTS dataset")

 There are 2177068 Rows and 58 Columns in HTS dataset


In [66]:
pd.crosstab(
    hts['EverTestedForHiv'],
    hts['TestedHIVBefore']
) #85% of the data match

TestedHIVBefore,No,Yes
EverTestedForHiv,,
No,281699,103829
Yes,22838,438559


In [67]:
#Check completeness:
hts['EverTestedForHiv'].isna().mean() #6.30%
hts['TestedHIVBefore'].isna().mean() #0.43%
#Drop EverTestedForHIV high missingness compared to TestedHIVBefore

np.float64(0.0043370257612532085)

In [68]:
#Keep variable...PatientDisabled. Drop Disability & DisabilityType because of high missingness
pd.crosstab(
    clients['PatientDisabled'],
    tests['FinalTestResult'],
    dropna=False
)

FinalTestResult,NEGATIVE,POSITIVE,NaN
PatientDisabled,,,
No,927642,25888,1
Yes,235632,6582,0
NaN,2718,87,0


Clean columns with _x & _y

In [69]:
#Identify the columns mismatch %
pairs = [
    ('CoupleDiscordant_x', 'CoupleDiscordant_y')
]

for x, y in pairs:
    match_pct = hts[x].fillna('NA').eq(hts[y].fillna('NA')).mean() * 100
    print(f"{x} vs {y}: {match_pct:.2f}%")


CoupleDiscordant_x vs CoupleDiscordant_y: 95.71%


In [70]:
#Checking conflicts for the couple dicordant data
pd.crosstab(
    hts['CoupleDiscordant_x'],
    hts['CoupleDiscordant_y'],
    dropna=False
)

CoupleDiscordant_y,No,Yes,NaN
CoupleDiscordant_x,,,
Declined to answer,0,0,18
No,293,101,17010
Yes,151,1215,30775
NaN,42262,3149,2082094


In [ ]:
#Download the conflict records
#conflicts_553 = hts[
  #  (
   #     (hts['CoupleDiscordant_x'] == 'Yes') &
    #    (hts['CoupleDiscordant_y'] == 'No')
    #)
    |
    #(
     #   (hts['CoupleDiscordant_x'] == 'No')&
      #  (hts['CoupleDiscordant_y'] == 'Yes')
    #)
#]

#conflicts_553.to_excel(
 #   'CoupleDiscordant_Conflicts.xlsx',
  #  index=False
#)

In [71]:
#Keep values from test table
hts['CoupleDiscordant'] = (
    hts['CoupleDiscordant_y']
    .combine_first(hts['CoupleDiscordant_x'])
)

hts.drop(
    columns=['CoupleDiscordant_x', 'CoupleDiscordant_y'],
    inplace=True
)

In [72]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType', 'Dob', 'Sex',
       'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs',
       'PriorityPopulationType', 'MFL_Code', 'Facility_Name', 'County',
       'SubCounty', 'Owner', 'Latitude', 'Longitude', 'SDP', 'SDP_Agency',
       'EMR', 'Project

In [73]:
# Identify missing or no-variance variables to exclude
# Get breakdown by variable
# Calculate missingness for all columns and display only those with >50% missing

missing_percent = hts.isnull().mean() * 100

high_missing = missing_percent[missing_percent > 50].sort_values(ascending=False)

print(high_missing)

Project                    100.000000
DisabilityType              99.782000
ResultOfHIVSelf             98.705599
PriorityPopulationType      98.278418
Disability                  98.142042
ViolenceScreeningType       97.542107
MothersStatus               97.403572
PriorityPopulation          97.032063
CoupleDiscordant            95.637527
KeyPopulation               93.028100
RelationshipWithContact     90.072382
NumberOfPartners            88.957120
MonthsSinceLastTest         83.205256
EverTestedForHiv            60.941734
ClientTestedAs              60.713216
FinalTestResult             59.597174
dtype: float64


In [74]:
print(type(hts))

<class 'pandas.core.frame.DataFrame'>


In [80]:
variables_to_drop = ["Project", "SDP_Agency",'Owner','Disability','DisabilityType','EverTestedForHiv','RelationshipWithContact' ]

hts= hts.drop(columns=variables_to_drop, errors="ignore")

#rows,cols = hts.shape
#print(f"There are {rows} rows and {cols} columns in the hts table")

In [81]:
print("Disability" in hts.columns)
print("DisabilityType" in hts.columns)

False
False


In [83]:
print(hts.columns.tolist())

['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation', 'PriorityPopulation', 'IsHealthWorker', 'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive', 'NewPartner', 'PartnerHIVStatus', 'MultiplePartners', 'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother', 'ExperiencedViolenceScreening', 'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus', 'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices', 'ViolenceScreeningType', 'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'MonthsSinceLastTest', 'FinalTestResult', 'ClientTestedAs', 'PriorityPopulationType', 'MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Latitude', 'Longitude', 'SDP', 'EMR', 'KEPH_Level', 'Ward', 'CoupleDiscordant']


In [84]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'TestedHIVBefore',
       'ResultOfHIV', 'EverHadSex', 'SexuallyActive', 'NewPartner',
       'PartnerHIVStatus', 'MultiplePartners', 'NumberOfPartners',
       'AlcoholSex', 'MoneySex', 'CondomBurst', 'UnknownStatusPartner',
       'KnownStatusPartner', 'Pregnant', 'BreastfeedingMother',
       'ExperiencedViolenceScreening', 'CurrentlyOnPrep',
       'TraditionalProcedures', 'MothersStatus', 'ResultOfHIVSelf',
       'ScreenedTB', 'TBStatus', 'ReceivedServices', 'ViolenceScreeningType',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'MonthsSinceLastTest',
       'FinalTestResult', 'ClientTestedAs', 'PriorityPopulationType',
       'MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Latitude',
       'Longitude', 'SDP', 'EMR', 'KEPH_Level', 'Ward', 'CoupleDiscordant'],
      dtype='object')

In [86]:
#download the hts file
hts.to_csv('hts1.csv', index=False)

from google.colab import files
files.download('hts1.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

                                 **   Data cleaning and feature generation **

In [ ]:
hts = pd.read_csv('hts.csv',low_memory=False)
#display(hts.head())
rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2179395 rows and 63 columns in the hts table


# Demographic Features

Age

In [115]:
# Create adult/child flag for checking patterns more easily
# Create VDate as date field, DOB as date field, and Age

hts['VDate'] = pd.to_datetime(hts['VisitDate'].astype(str).str[:10])
hts['DOB'] = pd.to_datetime(hts['Dob'].astype(str).str[:10], errors='coerce')

# Calculate Age
hts['Age'] = (hts['VDate'] - hts['DOB']).dt.days // 365

# Filter out erroneous ages (Age > 0 and Age < 100)
hts = hts[hts['Age'] > 0]
hts = hts[hts['Age'] < 100]

# Create cohort column
hts['cohort'] = np.where(hts['Age'] >= 15, "Adult", "Child")

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after age filtering")

There are 2176785 rows and 57 columns in the hts table after age filtering


Sex

In [116]:
#gender
#display(hts['Sex'].value_counts(dropna=False))

hts['Sex'] = hts['Sex'].astype(str).str.upper()
hts['Sex'] = hts['Sex'].str.replace('FEMALE', 'F')
hts['Sex'] = hts['Sex'].str.replace('MALE', 'M')

#display(hts['Sex'].value_counts(dropna=False))


In [117]:
#check positivity rate
tab = pd.crosstab(
    hts['Sex'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ positivity rate is higher in males

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
Sex,,,,
F,598530,14772,887930,2.408601
M,257682,8517,409354,3.199486


Marital Status

In [118]:
# MaritalStatus - Married, Polygamous, Separated/Divorced, Single, Minor
#display(hts['MaritalStatus'].value_counts(dropna=False))

hts['MaritalStatus'] = hts['MaritalStatus'].astype(str).str.upper()

conditions = [
    hts['Age'] < 15,
    hts['MaritalStatus'].str.contains('DIVORCED|SEPARATED', na=False),
    hts['MaritalStatus'].str.contains('POLYGAMOUS', na=False),
    (hts['MaritalStatus'].str.contains('NOT|NEVER|SINGLE|NO', na=False)) & (hts['Age'] >= 15),
    hts['MaritalStatus'].str.contains('MARRIED|LIVING|YES|COHABITING', na=False)
]

choices = [
    "MINOR",
    "DIVORCED",
    "POLYGAMOUS",
    "SINGLE",
    "MARRIED"
]

hts['MaritalStatus'] = np.select(conditions, choices, default=hts['MaritalStatus'])

#display(hts['MaritalStatus'].value_counts(dropna=False))

In [119]:
#Check positivity rate
tab = pd.crosstab(
    hts['MaritalStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# Positivity rate is higher in Divorced and widowed at 13%

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
MaritalStatus,,,,
DIVORCED,15927,2379,22650,12.995739
MARRIED,7549,350,8687,4.430941
MINOR,38351,688,53393,1.762340
NAN,111223,2235,298874,1.969892
OTHER,0,1,0,100.000000
POLYGAMOUS,19262,1169,26355,5.721697
SINGLE,655117,15086,873328,2.250960
WIDOWED,8783,1381,13997,13.587170


# Clinical Risk Features

Pregnancy

In [120]:
# Pregnant
hts['Pregnant'] = hts['Pregnant'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['Sex'].str.upper() == 'MALE',
    hts['Age'] > 50,
    hts['Pregnant'].str.contains('YES', na=False),
    hts['Pregnant'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'NR',
    'NR',
    'YES',
    'NO'
]

hts['Pregnant'] = np.select(conditions, choices, default='nan')

#display(hts['Pregnant'].value_counts(dropna=False))

In [121]:
#Check for positivity
tab = pd.crosstab(
    hts['Pregnant'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ low positivity rate amongst pregnant women

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
Pregnant,,,,
NO,424109,10189,605589,2.346085
NR,70679,3176,121396,4.300318
YES,121653,2331,196678,1.880081
nan,239771,7593,373621,3.069565


BreastfeedingMother

In [122]:
#BreastfeedingMother
hts['BreastfeedingMother'] = hts['BreastfeedingMother'].astype(str).str.upper()
conditions = [
    hts['Age'] <=9,
    hts['Sex'].str.upper() == 'MALE',
    hts['Age'] > 50,
    hts['BreastfeedingMother'].str.contains('YES', na=False),
    hts['BreastfeedingMother'].str.contains('NO', na=False),
    hts['BreastfeedingMother'].str.contains('DECLINED', na=False)
]

choices = [
    'NR',
    'NR',
    'NR',
    'YES',
    'NO',
    'DECLINED'
]
hts['BreastfeedingMother'] = np.select(conditions, choices, default='nan')
#display(hts['BreastfeedingMother'].value_counts(dropna=False))

In [123]:
#check positivity rate
tab = pd.crosstab(
    hts['BreastfeedingMother'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ low positivity rate amongst breastfeeding women

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
BreastfeedingMother,,,,
DECLINED,2553,56,3995,2.146416
NO,375858,11448,563626,2.955802
NR,70679,3176,121396,4.300318
YES,160030,858,218076,0.533290
nan,247092,7751,390191,3.041480


# **HIV Testing History Features**

 EverTestedForHiv

In [125]:
# Ever Tested for HIV

hts['TestedHIVBefore'] = hts['TestedHIVBefore'].astype(str).str.upper()

conditions = [
    hts['TestedHIVBefore'].str.contains('YES', na=False),
    hts['TestedHIVBefore'].str.contains('NO', na=False)
]

choices = [
    "YES",
    "NO"
]

hts['TestedHIVBefore'] = np.select(conditions, choices, default=hts['TestedHIVBefore'])

In [126]:
#Check for positivity rate
tab = pd.crosstab(
    hts['TestedHIVBefore'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) # Positivity rate is higher with those ever tested for HIV

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
TestedHIVBefore,,,,
NAN,3131,444,5865,12.419580
NO,301760,10834,507082,3.465837
YES,551321,12011,784337,2.132135


MonthsSinceLastTest

In [ ]:
# Months since last test
hts['MonthsSinceLastTest'] = pd.to_numeric(hts['MonthsSinceLastTest'], errors='coerce')

conditions = [
    hts['EverTestedForHiv'] != 'YES',
    (hts['MonthsSinceLastTest'] >= 0) & (hts['MonthsSinceLastTest'] <= 6),
    (hts['MonthsSinceLastTest'] >= 7) & (hts['MonthsSinceLastTest'] <= 12),
    (hts['MonthsSinceLastTest'] >= 13) & (hts['MonthsSinceLastTest'] <= 24),
    hts['MonthsSinceLastTest'] > 24
]

choices = [
    'NR',
    'LASTSIXMONTHS',
    'SEVENTOTWELVE',
    'ONETOTWOYEARS',
    'MORETHANTWOYEARS'
]

hts['MonthsSinceLastTest'] = np.select(conditions, choices, default='nan')


#display(hts['MonthsSinceLastTest'].value_counts(dropna=False))

In [ ]:
#Check positivity
tab = pd.crosstab(
    hts['MonthsSinceLastTest'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# higher positivity in those tested more than 2years ago at 6%

#Behaviour Risk Features

New Partner

In [ ]:
# New Partner

hts['NewPartner'] = hts['NewPartner'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['NewPartner'].str.contains('YES', na=False),
    hts['NewPartner'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['NewPartner'] = np.select(conditions, choices, default='nan')
#display(hts['NewPartner'].value_counts(dropna=False))

In [ ]:
#Check for positivity rate
tab = pd.crosstab(
    hts['NewPartner'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 4% positivity for those with new partners

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
NewPartner,,,,
NO,1202487,27294,35567,2.219420
NR,52108,1055,2045,1.984463
YES,436166,16998,4987,3.750960
nan,375799,8843,15760,2.299021


In [ ]:
#Check frequency of SexuallyActive vs New partner
cross_tab_sex_partner = pd.crosstab(hts['SexuallyActive'], hts['NewPartner'])
display(cross_tab_sex_partner) # Data to be reviewed are those not sexually active but have new patner

NewPartner,NO,NR,YES,nan
SexuallyActive,,,,
NAN,7191,0,5057,297104
NO,148343,0,11092,36357
NR,0,55208,0,0
YES,1109814,0,442002,66941


Multiple Partners

In [ ]:
# Number of  Partners
hts['NumberOfPartners'] = pd.to_numeric(hts['NumberOfPartners'], errors='coerce')

conditions = [
    hts['Age'] <= 9,
    (hts['NumberOfPartners'] >= 2) | (hts['MultiplePartners'] == 'YES'),
    hts['NumberOfPartners'] == 1
]

choices = [
    'NR',
    'MULTIPLE',
    'SINGLE'
]
hts['NumberOfPartners'] = np.select(conditions, choices, default='nan')

#display(hts['NumberOfPartners'].value_counts(dropna=False))

In [ ]:
#Check positivity rate
tab = pd.crosstab(
    hts['NumberOfPartners'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) # ~5% and 4.5% positivity amongst with Single and multiple partners

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
NumberOfPartners,,,,
MULTIPLE,295605,13914,2848,4.495362
NR,52108,1055,2045,1.984463
SINGLE,324,17,44,4.985337
nan,1718523,39204,53422,2.230380


In [ ]:
#Quick check Multiplepartner is Yes and Sexually active is No

nrow_multiple_partners_not_sexually_active = hts[(hts['MultiplePartners'] == 'YES') & (hts['SexuallyActive'] == 'NO')].shape[0]
print(f"Number of rows where MultiplePartners is 'YES' and SexuallyActive is 'NO': {nrow_multiple_partners_not_sexually_active}")

Number of rows where MultiplePartners is 'YES' and SexuallyActive is 'NO': 8328


In [ ]:
# Filter and count rows where MultiplePartners is 'YES' and NumberOfPartners is not 'MULTIPLE'
nrow_multiple_not_multiple_categorized = hts[
    (hts['MultiplePartners'] == 'YES') &
    (hts['NumberOfPartners'] != 'MULTIPLE')
].shape[0]

print(f"Number of rows where MultiplePartners is 'YES' and NumberOfPartners is not 'MULTIPLE': {nrow_multiple_not_multiple_categorized}")

Number of rows where MultiplePartners is 'YES' and NumberOfPartners is not 'MULTIPLE': 13


In [ ]:
# 1. MultiplePartners = YES but NumberOfPartners is not MULTIPLE
mask1 = (
    (hts['MultiplePartners'] == 'YES') &
    (hts['NumberOfPartners'] != 'MULTIPLE')
)

hts.loc[mask1, 'MultiplePartners'] = 'NR'

print(
    f"Corrected {mask1.sum()} rows: "
    "MultiplePartners set to 'NR'where it was 'YES' and NumberOfPartners was not 'MULTIPLE' due to age."
)

# 2. NumberOfPartners = MULTIPLE but EverHadSex = NO
mask2 = (
    (hts['NumberOfPartners'] == 'MULTIPLE') &
    (hts['EverHadSex'] == 'NO')
)

hts.loc[mask2, 'EverHadSex'] = 'YES'

print(
    f"Corrected {mask2.sum()} rows: "
    "EverHadSex set to 'YES'where NumberOfPartners was 'MULTIPLE' and EverHadSex was 'NO'."
)

Corrected 13 rows: MultiplePartners set to 'NR'where it was 'YES' and NumberOfPartners was not 'MULTIPLE' due to age.
Corrected 0 rows: EverHadSex set to 'YES'where NumberOfPartners was 'MULTIPLE' and EverHadSex was 'NO'.


MoneySex

In [ ]:
#Money Sex
hts['MoneySex'] = hts['MoneySex'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['MoneySex'].str.contains('YES', na=False),
    hts['MoneySex'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['MoneySex'] = np.select(conditions, choices, default='nan')

#display(hts['MoneySex'].value_counts(dropna=False))



In [ ]:
#Check for positivity rate
tab = pd.crosstab(
    hts['MoneySex'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~3.2% positivity rate for those who  have sex for money

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
MoneySex,,,,
NO,1390105,36062,37392,2.528596
NR,52108,1055,2045,1.984463
YES,218231,7147,2638,3.171117
nan,406116,9926,16284,2.385817


In [ ]:
# Calculate the number of rows where MoneySex is 'YES' and EverHadSex is 'NO'
num_money_sex_no_sex = hts[
    (hts['MoneySex'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where MoneySex is 'YES' and EverHadSex is 'NO': {num_money_sex_no_sex}")

Number of rows where MoneySex is 'YES' and EverHadSex is 'NO': 0


AlcoholSex

In [ ]:
# Alcohol Sex
hts['AlcoholSex'] = hts['AlcoholSex'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['AlcoholSex'].str.contains('ALWAYS', na=False),
    hts['AlcoholSex'].str.contains('SOMETIMES', na=False),
    hts['AlcoholSex'].str.contains('NEVER|NOT', na=False)
]

choices = [
    'NR',
    'ALWAYS',
    'SOMETIMES',
    'NEVER'
]

hts['AlcoholSex'] = np.select(conditions, choices, default='nan')

#display(hts['AlcoholSex'].value_counts(dropna=False))

In [ ]:
#check positivity rate
tab = pd.crosstab(
    hts['AlcoholSex'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~3.7% positivity rate for those who always have alcohol sex

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
AlcoholSex,,,,
ALWAYS,41625,1632,529,3.772800
NEVER,1382891,34439,37804,2.429850
NR,52108,1055,2045,1.984463
SOMETIMES,202352,7631,2150,3.634104
nan,387584,9433,15831,2.375969


CondomBurst

In [ ]:
#Condom Burst
hts['CondomBurst'] = hts['CondomBurst'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['CondomBurst'].str.contains('YES', na=False),
    hts['CondomBurst'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['CondomBurst'] = np.select(conditions, choices, default='nan')
#display(hts['CondomBurst'].value_counts(dropna=False))

In [ ]:
#check positivity rate
tab = pd.crosstab(
    hts['CondomBurst'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 3.3.% positivity rate for those who said they have had condom burst

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
CondomBurst,,,,
NO,1405839,36284,37977,2.516013
NR,52108,1055,2045,1.984463
YES,199851,6854,2184,3.315837
nan,408762,9997,16153,2.387292


In [ ]:
# Calculate the number of rows where CondomBurst is 'YES' and EverHadSex is 'NO'
num_condom_burst_no_sex = hts[
    (hts['CondomBurst'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where CondomBurst is 'YES' and EverHadSex is 'NO': {num_condom_burst_no_sex}")

Number of rows where CondomBurst is 'YES' and EverHadSex is 'NO': 0


# Partner Risk Features

PartnerHIVStatus

In [ ]:
# PartnerHIVStatus
hts['PartnerHIVStatus'] = hts['PartnerHIVStatus'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['PartnerHIVStatus'].str.contains('POSITIVE', na=False),
    hts['PartnerHIVStatus'].str.contains('NEGATIVE', na=False),
    hts['PartnerHIVStatus'].str.contains('UNKNOWN', na=False)
]

choices = [
    'NR',
    'POSITIVE',
    'NEGATIVE',
    'UNKNOWN'
]

hts['PartnerHIVStatus'] = np.select(conditions, choices, default='nan')
display(hts['PartnerHIVStatus'].value_counts(dropna=False))

,count
PartnerHIVStatus,
UNKNOWN,1049245
NEGATIVE,627884
nan,395573
NR,55208
POSITIVE,51199


In [ ]:
#Check for positivity rate
tab = pd.crosstab(
    hts['PartnerHIVStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 14% positivity for those  partnersHIV status was positive and 3% for those who did not know the partners status

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
PartnerHIVStatus,,,,
NEGATIVE,594091,4540,29253,0.758397
NR,52108,1055,2045,1.984463
POSITIVE,43747,6975,477,13.751429
UNKNOWN,1005222,33074,10949,3.185411
nan,371392,8546,15635,2.249314


In [ ]:
#Quick check on the shape of dataset
rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2179109 rows and 67 columns in the hts table


UnknownStatusPartner

In [ ]:
# Unprotected Sex with partner with unknown HIV status
hts['UnknownStatusPartner'] = hts['UnknownStatusPartner'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['UnknownStatusPartner'].str.contains('YES', na=False),
    hts['UnknownStatusPartner'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]
hts['UnknownStatusPartner'] = np.select(conditions, choices, default='nan')
#display(hts['UnknownStatusPartner'].value_counts(dropna=False))


In [ ]:
#Check positivity rate
tab = pd.crosstab(
    hts['UnknownStatusPartner'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) # ~4% positivity rate for those who do not knpw their partners status

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
UnknownStatusPartner,,,,
NO,1119123,23061,35776,2.019027
NR,52108,1055,2045,1.984463
YES,499054,20436,5233,3.933858
nan,396275,9638,15305,2.374400


In [ ]:
# Calculate the number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO'
num_USP_no_sex= hts[
    (hts['UnknownStatusPartner'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO': {num_USP_no_sex}")

Number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO': 0


KnownStatusPartner

In [ ]:
# Unprotected Sex with partner with known HIV status
hts['KnownStatusPartner'] = hts['KnownStatusPartner'].astype(str).str.upper

condtions = [
    hts['Age'] <= 9,
    hts['KnownStatusPartner'].str.contains('YES', na=False),
    hts['KnownStatusPartner'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['KnownStatusPartner'] = np.select(conditions, choices, default='nan')
#display(hts['KnownStatusPartner'].value_counts(dropna=False))

In [ ]:
#check positivity rate
tab = pd.crosstab(
    hts['KnownStatusPartner'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# ~ higher positivity amongst those who know their partners HIV status

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
KnownStatusPartner,,,,
NO,1119123,23061,35776,2.019027
NR,52108,1055,2045,1.984463
YES,499054,20436,5233,3.933858
nan,396275,9638,15305,2.374400


In [ ]:
# Calculate the number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO'
num_UP_no_sex= hts[
    (hts['KnownStatusPartner'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO': {num_UP_no_sex}")

Number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO': 0


Couple Discordant

In [ ]:
# Couple Discordant
hts['CoupleDiscordant'] = hts['CoupleDiscordant'].astype(str).str.upper()

conditions = [
    hts['MaritalStatus'].str.contains('SINGLE|MINOR', na=False), # If MaritalStatus is SINGLE or MINOR
    hts['CoupleDiscordant'].str.contains('YES', na=False),
    hts['CoupleDiscordant'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['CoupleDiscordant'] = np.select(conditions, choices, default='nan')
#display(hts['CoupleDiscordant'].value_counts(dropna=False))

 #most discodant couples are tested as individuals

In [ ]:
#Check positivity
tab = pd.crosstab(
    hts['CoupleDiscordant'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #8% positivity in discordant couples

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
CoupleDiscordant,,,,
NO,20908,2127,33,9.233775
NR,1559869,35309,42474,2.213483
YES,10672,952,93,8.189952
nan,475111,15802,15759,3.218900


# Key Population  Features

Population Type

In [ ]:
## Population Type -
# If Null, then NA
hts['PopulationType'] = hts['PopulationType'].fillna('NA')

display(hts['PopulationType'].value_counts(dropna=False))

,count
PopulationType,
General Population,1939875
Key Population,157105
Priority Population,70996
NA,11046
Vulnerable Population,90


In [ ]:
#Rename Vulnerable_Pop to Key_Pop because it has very few records
hts['PopulationType']= hts['PopulationType'].replace(
    'Vulnerable Population',
    'Key Population'
)
hts['PopulationType'].value_counts(dropna=False)

,count
PopulationType,
General Population,1939875
Key Population,157195
Priority Population,70996
NA,11046


In [ ]:
# Label as GP, KP, Priority

# Convert PopulationType to uppercase
hts['PopulationType'] = hts['PopulationType'].str.upper()

# Map PopulationType values to 'GP', 'KP', or 'PRIORITY'
conditions = [
    hts['PopulationType'].str.contains('GENERAL', na=False),
    hts['PopulationType'].str.contains('KEY', na=False),
    hts['PopulationType'].str.contains('PRIORITY', na=False),
    hts['PopulationType'].str.contains('VULNERABLE', na=False)
]
choices = ['GP', 'KP', 'PRIORITY', 'VP']
hts['PopulationType'] = np.select(conditions, choices, default=hts['PopulationType'])

Key Population

In [ ]:
display(hts['KeyPopulation'].value_counts(dropna=False))

,count
KeyPopulation,
NaN,2026906
Female sex worker,87363
Men who have sex with men,36615
People in prison and other closed settings,23884
People who inject drugs,4299
Other,45


In [ ]:
# We're going to use NR for not relevant throughout
# Clean labels for KPs. Create other for rare values

hts['KeyPopulation'] = hts['KeyPopulation'].astype(str).str.upper()

conditions = [
    hts['PopulationType'] != 'KP', # If PopulationType is not KP, set to NR
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('FEMALE', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('MEN', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('PRISON', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('DRUGS|OTHER', na=False),

]

choices = [
    'NR',
    'FSW',
    'MSM',
    'PRISONER',
    'PWID'
]

hts['KeyPopulation'] = np.select(conditions, choices, default=hts['KeyPopulation']) #Combined Other with PWID because Other had 45records

#display(hts['KeyPopulation'].value_counts(dropna=False))

display(hts[['PopulationType', 'KeyPopulation']].value_counts(dropna=False))

PopulationType  KeyPopulation
GP              NR               1939875
KP              FSW                87361
PRIORITY        NR                 70996
KP              MSM                36609
                PRISONER           23883
NA              NR                 11046
KP              NAN                 4998
                PWID                4344
Name: count, dtype: int64

In [ ]:
#Check for positivity rate

pd.crosstab(
    hts['KeyPopulation'],
    hts['FinalTestResult'],
    dropna=False
)


FinalTestResult,NEGATIVE,POSITIVE,NaN
KeyPopulation,,,
FSW,84354,794,2213
MSM,35762,495,352
NAN,4782,95,121
NR,1914084,52472,55358
PRISONER,23300,312,271
PWID,4278,22,44


Priority Population

In [ ]:
display(hts['PriorityPopulation'].value_counts(dropna=False))

,count
PriorityPopulation,
NaN,2114458
Adolescent and young girls,22991
Fisher folk,19191
Prisoner,18709
Truck driver,2130
Young women aged 15-24 years,1220
Military and other uniformed services,369
Families and children living on the streets,13
People who abuse alcohol and other drugs,13


In [ ]:
## Priority Population
#display(hts['PriorityPopulation'].value_counts(dropna=False))

hts['PriorityPopulation'] = hts['PriorityPopulation'].astype(str).str.upper()

conditions = [
    hts['PopulationType'] != 'PRIORITY', # If PopulationType is not PRIORITY, set to NR
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('FISHER', na=False),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('ADOLESCENT|YOUNG ', na=False),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('PRISONER', na=False),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('TRUCK', na=False),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('MILITARY|FAMILIES|PEOPLE|SERVICE|WIDOWS|OTHERS|ORPHANS', na=False)
]

choices = [
    'NR',
    'FISHERMEN',
    'AGYW',
    'PRISONER',
    'TRUCK',
    'OTHER'
]

hts['PriorityPopulation'] = np.select(conditions, choices, default=hts['PriorityPopulation'])

# Display the new distribution of PriorityPopulation
display(hts['PriorityPopulation'].value_counts(dropna=False))

,count
PriorityPopulation,
NR,2108113
AGYW,24175
FISHERMEN,19163
PRISONER,18698
NAN,6422
TRUCK,2129
OTHER,409


In [ ]:
#Check for positivity rate
pd.crosstab(
    hts['PriorityPopulation'],
    hts['FinalTestResult'],
    dropna=False
)

FinalTestResult,NEGATIVE,POSITIVE,NaN
PriorityPopulation,,,
AGYW,23789,163,223
FISHERMEN,18275,417,471
NAN,6333,38,51
NR,1997441,53239,57433
OTHER,390,13,6
PRISONER,18268,268,162
TRUCK,2064,52,13


In [ ]:
# Fix population type
hts_population_conditions = [
    (hts['KeyPopulation'] == 'NR') & (hts['PriorityPopulation'] == 'NR'),
    (hts['KeyPopulation'] != 'NR'),
    (hts['PriorityPopulation'] !='NR')

]

hts_population_choices = [
    'GP',
    'KP',
    'PRIORITY'
]

hts['PopulationType'] = np.select(
    hts_population_conditions,
    hts_population_choices,
    default=hts['PopulationType']
)

# Display the new distribution of PopulationType
display(hts['PopulationType'].value_counts(dropna=False)) # If  KP and Priority are NR, then they are GP, if they are Key Pop, classify as Kp same with Priority pop

,count
PopulationType,
GP,1950918
KP,157195
PRIORITY,70996


IsHealthCareWorker

In [ ]:
## Is Health Worker - what should 0 be? No or Null?
#display(hts['IsHealthWorker'].value_counts(dropna=False))

# If child, set to not relevant
hts['IsHealthWorker'] = hts['IsHealthWorker'].astype(str).str.upper()

conditions = [
    hts['cohort'] == 'Child',
    hts['IsHealthWorker'].str.contains('YES', na=False),
    hts['IsHealthWorker'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['IsHealthWorker'] = np.select(conditions, choices, default='NR')
display(hts['IsHealthWorker'].value_counts(dropna=False))

,count
IsHealthWorker,
NO,1950440
NR,141590
YES,87079


In [ ]:

#Check for positivity rate
tab = pd.crosstab(
    hts['IsHealthWorker'],
    hts['FinalTestResult']
)

tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
) * 100 #There is a very low positivity rates amongst healthcare workers~ 2.24%

# TB Risk Features

In [113]:
hts['TBStatus'].value_counts(dropna=False)

,count
TBStatus,
nan,2029455
PRESUMED,142589
CONFIRMED,4741


TB Status

In [107]:

#Convert TBStatus to uppercase then map values
hts['TBStatus'] = hts['TBStatus'].astype(str).str.upper()

conditions = [
    hts['TBStatus'].str.contains('SIGNS', na=False),
    hts['TBStatus'].str.contains('PRESUMED', na=False),
    hts['TBStatus'].str.contains('CONFIRMED', na=False),
    hts['TBStatus'].str.contains('TBSCREENING', na=False)
]

choices = [
    'NOTBSIGNS',
    'PRESUMED',
    'CONFIRMED',
    'NOTDONE'
]

hts['TBStatus'] = np.select(conditions, choices, default='nan')

In [114]:
#Check positivity rate
tab = pd.crosstab(
    hts['TBStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100

display(tab) # 12% positivity rate in those with confirmed TB Status

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
TBStatus,,,,
CONFIRMED,1570,223,2948,12.437256
PRESUMED,53184,5704,83701,9.686184
nan,801458,17362,1210635,2.120368


Screened TB

In [109]:
hts['ScreenedTB'].value_counts(dropna=False)

,count
ScreenedTB,
YES,1852352
NO,203519
Yes,61830
NaN,38384
No,16387
Declined to answer,4313


In [110]:
#Convert TBStatus to uppercase then map values
hts['ScreenedTB'] = hts['ScreenedTB'].astype(str).str.upper()

conditions = [
    hts['ScreenedTB'].str.contains('YES', na=False),
    hts['ScreenedTB'].str.contains('NO', na=False)
]

choices = [
    'YES',
    'NO'
]

hts['ScreenedTB'] = np.select(conditions, choices, default='nan')


In [112]:
#Check positivity rate
tab = pd.crosstab(
    hts['ScreenedTB'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100

display(tab) #High positivity rate in those who screened for TB

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
ScreenedTB,,,,
NO,82487,1756,135663,2.084446
YES,757600,20697,1135885,2.659268
nan,16125,836,25736,4.928955


# Violence and vulnerable Features

ExperiencedViolenceScreening

In [89]:
# Recently Experienced GBV
hts['ExperiencedViolenceScreening'] = hts['ExperiencedViolenceScreening'].astype(str).str.upper()


conditions_gbv = [
    hts['Age'] <= 9, # Not relevant for children
    hts['ExperiencedViolenceScreening'].str.contains('YES', na=False),
    hts['ExperiencedViolenceScreening'].str.contains('NO', na=False)
]

choices_gbv = [
    'NR',
    'YES',
    'NO'
]

hts['ExperiencedViolenceScreening'] = np.select(conditions_gbv, choices_gbv, default='nan')

# Create cross-tabulation tables (proportions)

prop_table_gbv_sex = pd.crosstab(hts['Sex'], hts['ExperiencedViolenceScreening'], normalize='index')
prop_table_age_gbv = pd.crosstab(hts['Age'], hts['ExperiencedViolenceScreening'], normalize='index')
prop_table_gbv_poptype = pd.crosstab(hts['PopulationType'], hts['ExperiencedViolenceScreening'], normalize='index')

#display(hts['ExperiencedViolenceScreening'].value_counts(dropna=False))



In [90]:
#Check for positivity
tab = pd.crosstab(
    hts['ExperiencedViolenceScreening'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab)# ~3% positivity rate in those who experienced violence

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
ExperiencedViolenceScreening,,,,
NO,737619,19298,1128864,2.549553
NR,21208,429,33165,1.982715
YES,79423,2674,109053,3.257123
nan,17962,888,26202,4.710875


ViolenceScreeningType

In [91]:
# Three values, create binaries for each

hts['ViolenceScreeningType'] = hts['ViolenceScreeningType'].astype(str).str.upper()


conditions_sexual = [
    hts['ViolenceScreeningType'].str.contains('SEXUAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_sexual = ['YES', 'NO', 'NO']
hts['GBVSexual'] = np.select(conditions_sexual, choices_sexual, default='NR')


conditions_physical = [
    hts['ViolenceScreeningType'].str.contains('PHYSICAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_physical = ['YES', 'NO', 'NO']
hts['GBVPhysical'] = np.select(conditions_physical, choices_physical, default='NR')


conditions_emotional = [
    hts['ViolenceScreeningType'].str.contains('EMOTIONAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_emotional = ['YES', 'NO', 'NO']
hts['GBVEmotional'] = np.select(conditions_emotional, choices_emotional, default='NR')


In [ ]:
crosstab_Emotional_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVEmotional'], dropna=False) # 8127 experienced VS but not GBVEmotional
crosstab_Physical_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVPhysical'], dropna=False) #16958 experienced VS but not GBVPhysical
crosstab_Sexual_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVSexual'], dropna=False) #2426 experienced VS but not GBVSexual

#Number of individuals who did not experience GBV (or status is NA) and  GBV types are 'YES': 0
#Number of individuals who experienced GBV but did not report any specific GBV type: 37504

In [ ]:
# Count rows where ExperiencedViolenceScreening is NA or 'NO', and no specific GBV type is 'YES'
num_rows_no_gbv_flags = hts[
    (hts['ExperiencedViolenceScreening'].isna() | (hts['ExperiencedViolenceScreening'] == 'NO')) &
    (hts['GBVEmotional'] == 'YES') &
    (hts['GBVPhysical'] == 'YES') &
    (hts['GBVSexual'] == 'YES')
].shape[0]

print(f"Number of individuals who did not experience GBV (or status is NA) and  GBV types are 'YES': {num_rows_no_gbv_flags}")

In [ ]:
#Number of individuals who experienced GBV but did not report any specific GBV type
num_inconsistent_gbv_screening = hts[
    (hts['ExperiencedViolenceScreening'] == 'YES') &
    (hts['GBVEmotional'] != 'YES') &
    (hts['GBVPhysical'] != 'YES') &
    (hts['GBVSexual'] != 'YES')
].shape[0]

print(f"Number of individuals who experienced GBV but did not report any specific GBV type: {num_inconsistent_gbv_screening}")

In [ ]:
#Quick check on the inconsistencies by Age
pd.cut(
   inconsistent_gbv['Age'],
    bins=[0,9,14,19,24,49,100]
).value_counts()

In [ ]:
#Quick check on the inconsistencies by positivity
inconsistent_gbv['FinalTestResult'] \
    .value_counts(normalize=True) \
    * 100

In [ ]:
#positivity rates for violence screening
tab = pd.crosstab(
    hts['ExperiencedViolenceScreening'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 3% positivity rate on those who experienced violence screening

PatientDisabled

In [ ]:
# PatientDisabled
hts['PatientDisabled'] = hts['PatientDisabled'].astype(str).str.upper()

conditions = [
    hts['PatientDisabled'].str.contains('NO', na=False),
    hts['PatientDisabled'].str.contains('YES', na=False)
]

choices = [
    "YES",
    "NO"
]

hts['PatientDisabled'] = np.select(conditions, choices, default=hts['PatientDisabled'])
#display(hts['PatientDisabled'].value_counts(dropna=False))

cross_tab_patient_disabled_population_type = pd.crosstab(hts['PatientDisabled'], hts['PopulationType']) #64376 are GP, 43141 are KP and 9817 are Priority

In [ ]:
#Check positivity rate
tab = pd.crosstab(
    hts['PatientDisabled'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #Positivity rate is lower in disabled patients

# PrEP Features

CurrentlyOnPrep

In [ ]:
#Currently on PrEP
hts['CurrentlyOnPrep'] = hts['CurrentlyOnPrep'].astype(str).str.upper()
conditions = [
    hts['Age'] <= 9,
    hts['CurrentlyOnPrep'].str.contains('YES', na=False),
    hts['CurrentlyOnPrep'].str.contains('NO', na=False)
]
choices = [
    'NR',
    'YES',
    'NO'
]
hts['CurrentlyOnPrep'] = np.select(conditions, choices, default='nan')
#display(hts['CurrentlyOnPrEP'].value_counts(dropna=False))
prep_by_sex = hts.groupby('Sex')['CurrentlyOnPrep'].value_counts(normalize=True).unstack(fill_value=0)


In [ ]:
#check positivity
tab = pd.crosstab(
    hts['CurrentlyOnPrep'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2% positivity rate amongst those not currently on PrEP

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
CurrentlyOnPrep,,,,
NO,1052255,26402,29157,2.447673
NR,52108,1055,2045,1.984463
YES,50859,347,689,0.677655
nan,911338,26386,26468,2.813834


ReceivedServices

In [ ]:
# Received Services - prep, pep, tb, sti
display(hts['ReceivedServices'])

,ReceivedServices
0,"PEP,PrEP,STI"
1,NaN
2,"PEP,PrEP,STI"
3,NaN
4,NaN
...,...
2179390,"PEP,PrEP,STI"
2179391,"PEP,PrEP,STI"
2179392,"PEP,PrEP,STI"
2179393,"PEP,PrEP,STI"


In [ ]:
# Four values, create binaries for each
hts['ReceivedServices'] = hts['ReceivedServices'].astype(str).str.upper()


def create_service_flag(df, service_col, keyword):
    conditions = [
        df[service_col].str.contains(keyword, na=False),
        ~df[service_col].isin(['NULL', '', 'NAN'])
    ]
    choices = ['YES', 'NO']
    return np.select(conditions, choices, default='NR')

# Create new binary columns for each service
hts['ReceivedPrEP'] = create_service_flag(hts, 'ReceivedServices', 'PREP') # 32K
hts['ReceivedPEP'] = create_service_flag(hts, 'ReceivedServices', 'PEP') # 30K
hts['ReceivedTB'] = create_service_flag(hts, 'ReceivedServices', 'TB') #33K
hts['ReceivedSTI'] = create_service_flag(hts, 'ReceivedServices', 'STI') #30K


In [ ]:
#Check positivity rate PrEP
tab = pd.crosstab(
    hts['ReceivedPrEP'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2.3% positivity rate for those who had received PrEP

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
ReceivedPrEP,,,,
NO,64393,2013,3909,3.031353
NR,878070,25136,24375,2.782975
YES,1124097,27041,30075,2.349067


In [ ]:
#Check positivity rate PEP
tab = pd.crosstab(
    hts['ReceivedPEP'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2.4% positivity rate for those who had received PEP

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
ReceivedPEP,,,,
NO,122580,2847,5286,2.269846
NR,878070,25136,24375,2.782975
YES,1065910,26207,28698,2.399651


In [ ]:
#check positivity rate for TB
tab = pd.crosstab(
    hts['ReceivedTB'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 4% of positivity rate for those who received TB services

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
ReceivedTB,,,,
NO,1166557,28119,31172,2.353693
NR,878070,25136,24375,2.782975
YES,21933,935,2812,4.088683


In [ ]:
#check positivity rate for STI
tab = pd.crosstab(
    hts['ReceivedSTI'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 2.4% positivity rate for those who received STI services


FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
ReceivedSTI,,,,
NO,119359,2392,5404,1.964666
NR,878070,25136,24375,2.782975
YES,1069131,26662,28580,2.433124


In [ ]:
#Quick Check
# no one currently on PrEP said they haven't received prep services
crosstab_prep_status = pd.crosstab(hts['ReceivedPrEP'], hts['CurrentlyOnPrep'], dropna=False)

# 24214 patients with presumed TB said they haven't received TB services
crosstab_TB_status = pd.crosstab(hts['ReceivedTB'], hts['TBStatus'], dropna=False)

# Sexual Exposure Risk Features

EverHadSex

In [ ]:
## For all sexual practice variables, if under 9 years old, then classify as NR

hts['EverHadSex'] = hts['EverHadSex'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['EverHadSex'].str.contains('YES', na=False),
    hts['EverHadSex'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['EverHadSex'] = np.select(conditions, choices, default='nan') # Changed default to 'nan' string

#display(hts['EverHadSex'].value_counts(dropna=False))

In [ ]:
#Check for positivity rate
tab= pd.crosstab(
    hts['EverHadSex'],
    hts['FinalTestResult'],
    dropna=False
)

tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #positivity rate of 2.6% for those who have ever had sex

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
EverHadSex,,,,
NO,2822,41,612,1.432064
NR,52108,1055,2045,1.984463
YES,1775699,47137,44386,2.585916
nan,235931,5957,11316,2.462710


Sexually Active

In [ ]:
# Sexually Active
#display(hts['SexuallyActive'].value_counts(dropna=False))


hts['SexuallyActive'] = hts['SexuallyActive'].astype(str).str.upper()
conditions = [
    hts['Age']<=9,
    hts['SexuallyActive'].str.contains('YES', na=False),
    hts['SexuallyActive'].str.contains('NO', na=False)
  ]


choices = [
            'NR',
            'YES',
            'NO'
 ]

hts['SexuallyActive']= np.select(conditions, choices, default='nan')
#display(hts['SexuallyActive'].value_counts(dropna=False))

In [ ]:
#Check for positivity rate
#make the options uppercase
hts['SexuallyActive']=hts['SexuallyActive'].astype(str).str.upper()

tab = pd.crosstab(
    hts['SexuallyActive'],
    hts['FinalTestResult'],
    dropna=False
)

tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) # ~2.7% positivity rate for those who have ever had sex and ~4% for those who declined to answer

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
SexuallyActive,,,,
NAN,288910,7281,13161,2.458211
NO,183192,3716,8884,1.988144
NR,52108,1055,2045,1.984463
YES,1542350,42138,34269,2.659408


In [ ]:
display(hts.columns)

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType', 'LoadDate_x',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'PriorityPopulationType',
       'MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agency', 'EMR', 'EMR_Status', 'HTS_

# Traditional Exposure Risk Features

TraditionalProcedures

In [ ]:
#Traditional Procedures
hts['TraditionalProcedures'] = hts['TraditionalProcedures'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 9,
    hts['TraditionalProcedures'].str.contains('YES', na=False),
    hts['TraditionalProcedures'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['TraditionalProcedures'] = np.select(conditions, choices, default='nan')
# display(hts['TraditionalProcedures'].value_counts(dropna=False))

In [ ]:
#check positivity rate
tab = pd.crosstab(
    hts['TraditionalProcedures'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ 3% positivity for those who had traditional procedures

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
TraditionalProcedures,,,,
NO,1838579,46049,48773,2.443400
NR,57015,1151,2395,1.978819
YES,94736,3062,1473,3.130943
nan,76230,3928,5718,4.900322


# Family Risk Features

MothersStatus

In [ ]:
#Mother's HIV Status - asked of kids only
hts['MothersStatus'] = hts['MothersStatus'].astype(str).str.upper()

conditions = [
    hts['Age'] >= 15,
    hts['MothersStatus'].str.contains('POSITIVE', na=False),
    hts['MothersStatus'].str.contains('NEGATIVE', na=False),
    hts['MothersStatus'].str.contains('UNKNOWN', na=False)
]

choices = [
    'NR',
    'POSITIVE',
    'NEGATIVE',
    'UNKNOWN'
]

hts['MothersStatus'] = np.select(conditions, choices, default='nan')
# display(hts['MothersStatus'].value_counts(dropna=False))

In [ ]:
#check positivity rates
tab = pd.crosstab(
    hts['MothersStatus'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #~ higher positivity rate amongst those whose mothers status was positive

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
MothersStatus,,,,
NEGATIVE,16418,51,1254,0.309673
NR,1978498,52589,55141,2.589205
POSITIVE,27905,876,456,3.043675
UNKNOWN,9007,150,360,1.638091
nan,34732,524,1148,1.486272


# Distribution by day of week

In [ ]:
#Let day of week (skip month because we don't have a whole year or multiple years)
hts['dayofweek'] = hts['VDate'].dt.day_name().str.upper() # Monday=0, Sunday=6

display(hts['dayofweek'].value_counts(dropna=False))

,count
dayofweek,
TUESDAY,448722
MONDAY,438571
WEDNESDAY,422345
THURSDAY,416552
FRIDAY,347544
SATURDAY,64628
SUNDAY,40747


In [ ]:
#Check positivity by day of week
tab = pd.crosstab(
    hts['dayofweek'],
    hts['FinalTestResult'],
    dropna=False
)
tab['PositivityRate'] = (
    tab['POSITIVE'] /
    (tab['POSITIVE'] + tab['NEGATIVE'])
)*100
display(tab) #positivity rate higher on Monday where majority of teste are done

FinalTestResult,NEGATIVE,POSITIVE,NaN,PositivityRate
dayofweek,,,,
FRIDAY,330406,7785,9353,2.301954
MONDAY,414230,13099,11242,3.065320
SATURDAY,62260,645,1723,1.025356
SUNDAY,39163,310,1274,0.785347
THURSDAY,395052,9964,11536,2.460150
TUESDAY,425055,11896,11771,2.722502
WEDNESDAY,400394,10491,11460,2.553269


In [ ]:
hts.columns

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Disability', 'DisabilityType', 'LoadDate_x',
       'Dob', 'Sex', 'MaritalStatus', 'PatientDisabled', 'EverTestedForHiv',
       'MonthsSinceLastTest', 'FinalTestResult', 'PriorityPopulationType',
       'MFL_Code', 'Facility_Name', 'County', 'SubCounty', 'Owner', 'Latitude',
       'Longitude', 'SDP', 'SDP_Agency', 'EMR', 'EMR_Status', 'HTS_

In [ ]:
cols_to_drop = ["Dob", "DOB", "present", "cohort", "VDate",
                "CurrentlyOnPep","DateTestedProvider",  'Disability', 'DisabilityType','LoadDate_x','KMPDC_reg_no','ReceivedPEP','EMR_Status', 'HTS_Use',
                'HTS_Deployment', 'Project', 'LoadDate_y', 'InfrastructureType',
                'KEPH_Level', 'KMPDC_reg_no', 'Ward',]

hts = hts.drop(columns=cols_to_drop, errors='ignore')

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after dropping columns") #There are 2179109 rows and 58 columns in the hts table after dropping columns

There are 2179109 rows and 58 columns in the hts table after dropping columns


In [ ]:
#download the hts file
hts.to_csv('hts_cleaned.csv', index=False)

from google.colab import files
files.download('hts_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
display(hts.columns)

Index(['SiteCode', 'PatientPk', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Sex', 'MaritalStatus', 'PatientDisabled',
       'EverTestedForHiv', 'MonthsSinceLastTest', 'FinalTestResult',
       'PriorityPopulationType', 'MFL_Code', 'Facility_Name', 'County',
       'SubCounty', 'Owner', 'Latitude', 'Longitude', 'SDP', 'SDP_Agency',
       'EMR', 'CoupleDiscordant', 'Age', 'ReceivedPrEP', 'ReceivedTB',
       'Receive

In [ ]:
#Check missingness in hts dataset
missing_values = hts.isnull().sum()
missing_values_percentage = (missing_values / len(hts)) * 100
missing_values_df = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_values_percentage})
missing_values_df

,Missing Values,Percentage
SiteCode,0,0.000000
PatientPk,0,0.000000
VisitDate,0,0.000000
PopulationType,0,0.000000
KeyPopulation,0,0.000000
PriorityPopulation,0,0.000000
IsHealthWorker,0,0.000000
RelationshipWithContact,1962670,90.067546
TestedHIVBefore,9440,0.433205
ResultOfHIV,930752,42.712503


In [ ]:
#Check for variance
cols = [
       'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'IsHealthWorker', 'RelationshipWithContact',
       'TestedHIVBefore', 'ResultOfHIV', 'EverHadSex', 'SexuallyActive',
       'NewPartner', 'PartnerHIVStatus', 'MultiplePartners',
       'NumberOfPartners', 'AlcoholSex', 'MoneySex', 'CondomBurst',
       'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'CurrentlyOnPrep', 'TraditionalProcedures', 'MothersStatus',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'ReceivedServices',
       'ViolenceScreeningType', 'Sex', 'MaritalStatus', 'PatientDisabled',
       'EverTestedForHiv', 'MonthsSinceLastTest', 'FinalTestResult',
       'PriorityPopulationType',  'CoupleDiscordant', 'Age', 'ReceivedPrEP', 'ReceivedTB',
       'ReceivedSTI', 'GBVSexual', 'GBVPhysical', 'GBVEmotional', 'dayofweek'
]

for col in cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(hts[col].value_counts(dropna=False, normalize=True) * 100)


Column: PopulationType
PopulationType
GP          89.528243
KP           7.213728
PRIORITY     3.258029
Name: proportion, dtype: float64

Column: KeyPopulation
KeyPopulation
NR          92.786272
FSW          4.009024
MSM          1.679999
PRISONER     1.095998
NAN          0.229360
PWID         0.199348
Name: proportion, dtype: float64

Column: PriorityPopulation
PriorityPopulation
NR           96.741971
AGYW          1.109398
FISHERMEN     0.879396
PRISONER      0.858057
NAN           0.294708
TRUCK         0.097700
OTHER         0.018769
Name: proportion, dtype: float64

Column: IsHealthWorker
IsHealthWorker
NO     89.506307
NR      6.497610
YES     3.996083
Name: proportion, dtype: float64

Column: RelationshipWithContact
RelationshipWithContact
NaN                                             90.067546
Sexual Contact                                   5.522486
Social Contact                                   3.709957
Sexual Contact,Social Contact                    0.359046
Needle 

In [ ]:
cols_to_drop = ['MFL_Code']

hts = hts.drop(columns=cols_to_drop, errors='ignore')

# Data Imputation

                    Let's join with GIS variables

In [ ]:
!pip install pyreadr
import pyreadr

In [ ]:
# Read the rds dataset
result = pyreadr.read_r('gis_features_iit.rds')
gis = result[None]

display(gis.head())

In [ ]:
if 'Latitude' in gis.columns:
    gis = gis.drop(columns=['Latitude'])
if 'Longitude' in gis.columns:
    gis = gis.drop(columns=['Longitude'])

    gis.drop_duplicates(subset=['FacilityCode'], keep='first', inplace=True)

    # Select only numeric columns for mean imputation
    numeric_cols = gis.select_dtypes(include=np.number).columns
    gis[numeric_cols] = gis[numeric_cols].fillna(gis[numeric_cols].mean())

    hts['SiteCode'] = hts['SiteCode'].astype(str)

    hts = pd.merge(hts, gis, left_on="SiteCode", right_on="FacilityCode", how="inner").drop(columns=['SiteCode_y'])

    hts.replace("", np.nan, inplace=True)

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after joining with GIS variables")

                                   Missing data imputation

In [ ]:
# We'll create two versions of the dataset
# 1) Keep missing values missing. Some ML models have sophisticated in-built ways of
# dealing with missing values, particularly XGBoost. For others, we'll need to impute.
# 2) Simple imputation - mean and mode. However, we'll only do this for variables that
# are present at least half the time. For very sparse variables, we don't have enough
# of a basis to impute. For these, we'll give missing values a label of MISSING


In [ ]:
# For all imputation, we're going to learn how to impute from the train set only to
# avoid any leakage. So, first step is to split dataset into train-eval-test.
# do 60-20-20 train-val-test split


In [ ]:
import random

random.seed(2231)
np.random.seed(2231)

In [ ]:
from sklearn.model_selection import train_test_split

# Drop rows where 'FinalTestResult' is NaN before splitting for stratification..
hts_cleaned = hts.dropna(subset=['FinalTestResult']).copy()

hts_train, hts_temp_test = train_test_split(hts_cleaned, test_size=0.4, random_state=2231, stratify=hts_cleaned['FinalTestResult'])
hts_val, hts_test = train_test_split(hts_temp_test, test_size=0.5, random_state=2231, stratify=hts_temp_test['FinalTestResult'])

print(f"Original HTS shape: {hts.shape}")
print(f"HTS shape after dropping NaNs in FinalTestResult (for splitting): {hts_cleaned.shape}")
print(f"Training set shape: {hts_train.shape}")
print(f"Validation set shape: {hts_val.shape}")
print(f"Test set shape: {hts_test.shape}")

In [ ]:
# Create sparse versions of the datasets by simply copying the split DataFrames
sparse_train_df = hts_train.copy()
sparse_val_df = hts_val.copy()
sparse_test_df = hts_test.copy()

# Store them in a dictionary for easy access, similar to the R list structure
sparse_datasets = {
    "sparse_train": sparse_train_df,
    "sparse_val": sparse_val_df,
    "sparse_test": sparse_test_df
}

print("Sparse datasets:")
for name, df in sparse_datasets.items():
    print(f"  {name} shape: {df.shape}")

In [ ]:
## Next, simple imputation
# First, identify which variables are present > 50% of the time and should be imputed
# Identify which variables are too sparse and instead will be given a value of MISSING
cols_to_impute = []
cols_to_unknown = []

selected_column_names = list(hts.columns[0:35]) + list(hts.columns[36:48])

for col_name in selected_column_names:
    vals = hts[col_name]

    non_missing_percentage = 100 * (vals.notna().sum() / len(vals))

    #print(col_name)
    #print(round(non_missing_percentage))

    # Condition for cols_to_impute: >50% non-missing AND has some missing values
    if non_missing_percentage > 50 and vals.isna().any():
        cols_to_impute.append(col_name)

    # Condition for cols_to_unknown: >50% missing values
    if (100 - non_missing_percentage) > 50:
        cols_to_unknown.append(col_name)

print(f"\nColumns to impute: {cols_to_impute}")
print(f"Columns to label as unknown: {cols_to_unknown}")

In [ ]:
# This function identifies the mode in one dataframe and imputes to another
# This is critical because we want the mode from the training set only to avoid leakage

def mode_excluding_nr(series):

    filtered_series = series[series != 'NR'].dropna()
    if not filtered_series.empty:
        return filtered_series.mode()[0]
    else:
        return 'UNKNOWN'

def replace_with_mode(df_calc, df_impute, col_name):
    imputation_mode = mode_excluding_nr(df_calc[col_name])

    if pd.api.types.is_numeric_dtype(df_impute[col_name]):
        df_impute[col_name] = df_impute[col_name].fillna(imputation_mode)
    else:
        df_impute[col_name] = df_impute[col_name].fillna(imputation_mode)
    return df_impute

train_simple_py = sparse_datasets['sparse_train'].copy()
val_simple_py = sparse_datasets['sparse_val'].copy()
test_simple_py = sparse_datasets['sparse_test'].copy()

for col_name in cols_to_impute:
    #print(f"  Imputing column: {col_name}")

    train_simple_py = replace_with_mode(train_simple_py, train_simple_py, col_name)
    val_simple_py = replace_with_mode(train_simple_py, val_simple_py, col_name) # Uses mode from train_simple_py
    test_simple_py = replace_with_mode(train_simple_py, test_simple_py, col_name) # Uses mode from train_simple_py

simple = {
    "simple_train": train_simple_py,
    "simple_val": val_simple_py,
    "simple_test": test_simple_py
}

for name, df_imp in simple.items():
    print(f"Missing values in {name} after translation imputation (first 10):\n{df_imp.isnull().sum().head(10)}")


In [ ]:
# Replace nan with Missing
for col_name in cols_to_unknown:
    # Iterate through all three datasets (train, val, test) and replace NaNs with 'MISSING'
    for df_key in simple:
        simple[df_key][col_name] = simple[df_key][col_name].fillna('MISSING')

#display(simple['simple_train'].head())

In [ ]:
# For simple imputations, add binary variables to indicate whether
# value was missing. This is so that we retain information about missingness.

for col_name in cols_to_impute + cols_to_unknown:
    for df_key, original_df in zip(simple.keys(), [hts_train, hts_val, hts_test]):
        simple[df_key][f'{col_name}_IS_MISSING'] = original_df[col_name].isna().astype(int)


In [ ]:
# For simple, add binary variables to indicate whether
#value was missing. This is so that we retain information about missingness.

sparse_cols_from_train = hts_train.columns[hts_train.isnull().any()].tolist()

# Process simple_train
sparse_binary_train = hts_train[sparse_cols_from_train].notna().astype(int)
sparse_binary_train .columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_train  = sparse_binary_train .reindex(simple['simple_train'].index)
simple['simple_train'] = pd.concat([simple['simple_train'], sparse_binary_train], axis=1)

# Do the same for val set
sparse_binary_val = hts_val[sparse_cols_from_train].notna().astype(int)
sparse_binary_val.columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_val = sparse_binary_val.reindex(simple['simple_val'].index)
simple['simple_val'] = pd.concat([simple['simple_val'], sparse_binary_val], axis=1)

# Repeat for test set
sparse_binary_test = hts_test[sparse_cols_from_train].notna().astype(int)
sparse_binary_test.columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_test = sparse_binary_test.reindex(simple['simple_test'].index)
simple['simple_test'] = pd.concat([simple['simple_test'], sparse_binary_test], axis=1)



### Saving Processed DataFrames to CSV

In [ ]:
# download sparse datasets to a CSV file
for name, df in sparse_datasets.items():
    file_name = f"{name}.csv"
    df.to_csv(file_name, index=False)
    print(f"Saved {name} to {file_name}")

print("All sparse datasets saved as CSVs.")

In [ ]:
# Download Sparse datasets to a CSV file
for name, df in simple.items():
    file_name = f"{name}.csv"
    df.to_csv(file_name, index=False)
    print(f"Saved {name} to {file_name}")

print("All simple imputed datasets saved as CSVs.")